# EXP-2026-007 / Q5-D — order-preserving beat identity join (quest54)

## `IMPLEMENTED — FULL RESULT NOT RUN`
## `EXP-2026-007 SCIENTIFIC RESULT NOT RUN`

### 이 notebook 의 상태 — 먼저 읽을 것

- **구현 승인과 실제 실행 승인은 별개의 승인이다.** 둘 다 2026-08-10 에 받았다.
  그러나 **이 파일은 미실행 상태로 커밋됐다** — 출력 셀이 전부 비어 있고,
  아래 어떤 표·그래프도 지금은 값을 담고 있지 않다. Colab 에서 순서대로 실행하면
  그때 채워진다. 여기에 적힌 숫자를 결과로 인용하면 안 된다.
- **아직 승인되지 않은 것**: V10 probability 값 열람 · association 분석 ·
  S PR-AUC · 모델 학습. join 이 자체 gate 를 통과해도 **또 다른 별도 승인** 전까지
  봉인이다. DS2 per-beat class label 은 DS1 규칙이 동결된 뒤 support gate 에서만 열린다.
- **실행 순서는 hash preflight → STOP/PASS → 실제 join 이다.** 자재 계약(hash)이
  닫히지 않으면 `JOIN_INPUT_ABSENT` 로 **매칭 전에** 멈춘다. 이건 join 성능이 아니라
  입력 계약의 문제이고, 우회하지 않는다.
- 결과 셀은 전부 `decision.json` · `null_summary.json` · `bootstrap.json` · `*.csv`
  를 읽는다. 숫자를 셀 안에서 다시 계산해 옮겨 적지 않는다
  (CLAUDE.md 「실행 로그 루프」 — 낡은 노트북으로 결과를 추측하지 않는다).

### 지금 알려진 미해결 자재 계약 3건 (셀 5 가 판정한다)

1. `mamba_data.npz` 는 등록 hash `b1c16106…` 과 **대조된 적이 없고**, Drive 에
   **같은 크기 사본이 3개**다(둘은 2026-08-10 생성). 바이트로 정본을 확정해야 한다.
2. V9·V10 캐시 hash 미계산.
3. `cache_v15b/mitdb/meta.json` 의 Drive 사본 미확인.

### 판정 후보

`JOIN_INPUT_ABSENT` · `JOIN_RULE_FALSIFIED` · `JOIN_SELECTION_BIASED` ·
`JOIN_UNRESOLVED` · `JOIN_IDENTIFIABLE` — 실행 전 상태는 `JOIN_RESULT_NOT_RUN`.

### 목차

1. 환경 및 승인 gate · 2. 입력 asset/hash 확인 (**preflight STOP/PASS**) ·
3. 44-record ledger 표 · 4. synthetic fixture 결과 · 5. Leg 1 replay audit ·
6. Leg 2 record-wise join · 7. DS1 gate report · 8. DS2 frozen gate report ·
9. negative-control/null plots · 10. class·record coverage plots ·
11. ambiguous/unmatched 원인표 · 12. equal-count 36 vs mismatch 8 진단 비교 ·
13. record 105·111·116·208·222 상세표 ·
14. record 232 S-share 원분포 대 certification 후 inflation ·
15. decision tree 최종 판정 · 16. 사람이 읽을 수 있는 해석 요약 ·
17. Drive bundle 저장 및 ingest 단계

## 1. 환경 및 승인 gate

이 셀들은 **아무 등록 자산도 열지 않는다.** `DESIGN` 과 `SYNTHETIC_FIXTURES` 는
합성 데이터만 쓰고, `JOIN_REPORT` 는 이미 만들어진 bundle 을 다시 읽을 뿐이다.
나머지 네 mode 는 모듈이 **파일을 열기 전에** 별도 실행 승인을 요구하며 중단한다.

In [ ]:
# ── 셀 1: 실행 설정 (정확히 하나의 mode) ─────────────────────────────────────
VALID_MODES = ("DESIGN", "SYNTHETIC_FIXTURES", "HASH_PREFLIGHT",
               "LEG1_REPLAY_AUDIT", "LEG2_RECORD_JOIN", "DS1_GATE",
               "DS2_GATE", "JOIN_REPORT")
MODE = "DESIGN"
assert MODE in VALID_MODES, f"MODE must be one of {VALID_MODES}"

# 순서: DESIGN -> SYNTHETIC_FIXTURES -> HASH_PREFLIGHT -> (PASS 여야) ->
#       LEG1_REPLAY_AUDIT -> LEG2_RECORD_JOIN -> DS1_GATE -> DS2_GATE -> JOIN_REPORT

# 병합된 코드를 쓴다. 아직 안 병합된 PR 을 돌려 보려면 그 PR 의 브랜치 이름을
# 정확히 넣는다(로컬 이름이 아니라 origin 에 실제로 있는 이름이어야 한다).
BRANCH = "main"
NEED_TESTS = 500          # 회귀 테스트 최소 통과 수
NEED_MODULE_VERSION = 3   # 이보다 낮으면 낡은 clone 이다 (셀 2 가 막는다)

# 등록 자산을 여는 것은 명시적 opt-in 이다. 사용자가 실행을 승인했으므로 True 로
# 둘 수 있지만, 값이 True 인 셀을 실행하는 순간 실제 데이터를 연다는 뜻이다.
OPEN_REGISTERED_DATA = False

# ── Drive 경로 (셀 3 에서 마운트한 뒤 사용) ─────────────────────────────────
DRIVE = "/content/drive/MyDrive"
ASSET_MITDB   = f"{DRIVE}/MedKOS/ecg-model/assets/EXP-2026-007_prep_data/source/mitdb-1.0.0"
# 사본 전수 비교. 등록본이 우선이고, 나머지 hash 일치본은
# byte_identical_duplicate 로 기록된다(중복은 ambiguity 가 아니다).
# 경로는 best-effort 다 — 존재하지 않는 후보는 absent 로 기록될 뿐 실패가 아니다.
# 일치본이 0개일 때만 JOIN_INPUT_ABSENT 다.
ASSET_MAMBA_CANDIDATES = [
    f"{DRIVE}/mitbih/mamba_data.npz",                       # 등록본 1p3HvC_…
    f"{DRIVE}/mitbih/kinkmap/v13pkg/mamba_data.npz",        # 1_Wg_7wH…
    f"{DRIVE}/mitbih/v9pkg/kinkmap/v13pkg/mamba_data.npz",  # 1hLhMp1z…
]
# result NPZ 는 DS2 산출물이다. DS1 에 같은 파일을 들이대면 행 수가 맞을 리 없어
# 정상 자산도 STOP 한다 → DS1 은 cache/ledger 로, DS2 는 result 25개 전수로 검증.
# (arm x seed grid 는 셀 5 에서 BJ.V10_RESULT_ARMS 로 읽는다. 이 셀은 아직
#  모듈을 import 하기 전이라 BJ 를 참조하면 안 된다.)
DS1_FROZEN_BUNDLE = ""   # DS2_GATE 때: 동결된 DS1 run bundle 경로
ASSET_CACHE_V9  = f"{DRIVE}/mitbih/v9~v13/v9/cache/mitdb"
ASSET_CACHE_V10 = f"{DRIVE}/mitbih/v9~v13/v10/cache/mitdb"
ASSET_V9_RESULTS  = f"{DRIVE}/mitbih/baseline_pkgs/v9pkg_results"
ASSET_V10_RESULTS = f"{DRIVE}/mitbih/baseline_pkgs/v10pkg_results"

RUN_DIR = ""      # 실행 셀이 timestamp 로 만든다. JOIN_REPORT 때는 기존 경로를 넣는다.
print("MODE =", MODE, "· OPEN_REGISTERED_DATA =", OPEN_REGISTERED_DATA)

In [ ]:
# ── 셀 1b: 런타임 설치 — Colab 재시작 때마다 이 셀부터 다시 돌린다 ──────────
# Colab 런타임이 끊기거나 재시작되면 pip 설치가 통째로 날아간다. 그래서 설치를
# repo 준비와 분리해 맨 앞에 둔다. 이 셀만 다시 돌리면 복구된다.
# 여기서 안 깔면 Leg 1 이 첫 .atr 에서, Leg 2 는 다 끝난 뒤 bundle 쓰는 자리에서
# 멈춘다(멈추는 것 자체는 설계대로다 — 부분 결과를 남기지 않는다).
import subprocess, sys

PIP_SPEC = ("wfdb==4.3.1", "pyarrow")   # wfdb 는 등록 런타임 버전에 고정
print("설치 중:", " ".join(PIP_SPEC))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_SPEC],
               check=True)

_ok = True
for _name in ("numpy", "wfdb", "pyarrow"):
    try:
        _m = __import__(_name)
        print(f"  OK   {_name:<10} {getattr(_m, '__version__', '?')}")
    except ImportError as _e:
        _ok = False
        print(f"  MISS {_name:<10} {_e}")
assert _ok, "설치 후에도 import 가 안 된다 — 런타임을 재시작하고 이 셀을 다시 돌린다."
print("런타임 준비 완료. 재시작하면 이 셀부터 다시 실행한다.")

In [ ]:
# ── 셀 2: repo 준비 + commit SHA + 회귀 테스트 ───────────────────────────────
import os, subprocess, sys

BRANCH = globals().get("BRANCH", "main")
MODE = globals().get("MODE", "DESIGN")
NEED_TESTS = int(globals().get("NEED_TESTS", 300))

# 고정 버전을 먼저 설치한다. 이게 없으면 Leg 1 이 첫 .atr 에서, 혹은 더 나쁘게
# Leg 2 가 다 끝난 뒤 bundle 쓰는 자리에서 죽는다.
PIP_SPEC = ("wfdb==4.3.1", "pyarrow")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_SPEC],
               check=True)

REPO = "/content/my-github-test"
URL = "https://github.com/ehdbddl06001-ui/my-github-test.git"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "50", URL, REPO], check=True)

fetched = subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH],
                         capture_output=True, text=True)
if fetched.returncode != 0:
    listed = subprocess.run(["git", "ls-remote", "--heads", URL],
                            capture_output=True, text=True).stdout
    names = sorted(l.split("refs/heads/")[-1] for l in listed.splitlines()
                   if "refs/heads/" in l)
    raise SystemExit(
        f"origin 에 '{BRANCH}' 브랜치가 없다. 셀 1 의 BRANCH 를 고쳐라.\n"
        f"origin 의 브랜치 목록:\n  " + "\n  ".join(names))

# -B + FETCH_HEAD: 로컬 브랜치가 이미 있든 없든 방금 받은 커밋으로 맞춘다.
subprocess.run(["git", "-C", REPO, "checkout", "-B", BRANCH, "FETCH_HEAD"],
               check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("branch:", BRANCH, "· commit:", COMMIT)

sys.path.insert(0, os.path.join(REPO, "mit-bih"))

# import 는 sys.modules 에 캐시된다 → git 으로 새 코드를 받아도 이 셀을 다시
# 돌리는 것만으로는 갱신되지 않는다. 낡은 모듈로 새 notebook 을 돌리면
# 원인을 알기 어려운 STOP 이 난다. 매번 캐시를 비우고 다시 import 한다.
for _name in [m for m in list(sys.modules)
              if m == "q5d_order_preserving_beat_join"]:
    del sys.modules[_name]
import q5d_order_preserving_beat_join as BJ
print("module file   :", BJ.__file__)
print("module version:", BJ.MODULE_VERSION, BJ.MODULE_BUILD)
print("rule fingerprint:", BJ.rule_fingerprint())

# 버전 숫자만으로는 내가 올리는 걸 잊으면 뚫린다 → notebook 이 실제로 쓰는
# 이름을 직접 확인한다. 새 셀이 새 함수를 쓰면 여기 이름을 추가하면 된다.
NEED_ATTRS = ("build_preflight", "assert_preflight_passed", "run_true_join",
              "resolve_canonical_mamba", "hash_file_set",
              "verify_against_publisher_checksums", "describe_checksum_mismatch",
              "release_ds2_support_gate", "verify_cache_ledger_contract",
              "V10_RESULT_ARMS", "MITDB_REGISTERED_FILE_COUNT",
              "NullContext", "run_null_shards", "finalize_null_shards",
              "finalize_ds1_gate", "NULL_RUNNER_VERSION")
_absent = [a for a in NEED_ATTRS if not hasattr(BJ, a)]
assert not _absent, (
    f"모듈에 {_absent} 가 없다 — 낡은 clone 이다. "
    f"BRANCH={BRANCH} (commit {COMMIT[:8]}) 에 최신 수정이 없다.")

NEED_MODULE_VERSION = int(globals().get("NEED_MODULE_VERSION", 1))
assert BJ.MODULE_VERSION >= NEED_MODULE_VERSION, (
    f"모듈 버전 {BJ.MODULE_VERSION} < 필요 {NEED_MODULE_VERSION}.\n"
    f"BRANCH={BRANCH} (commit {COMMIT[:8]}) 에 최신 수정이 없다. "
    f"병합이 끝났는지 확인하거나 BRANCH 를 해당 PR 브랜치로 바꾼다.")

# 이 MODE 가 실제로 쓰는 import 를 지금 확인한다 — 단계 중간이나 끝이 아니라.
_dep = BJ.check_runtime_dependencies(MODE)
for _row in _dep["dependencies"]:
    _mark = "OK  " if _row["available"] else "MISS"
    _reg = _row.get("registered_version")
    _note = ("" if not _reg else
             f" (등록 런타임 {_reg}"
             f"{'' if _row.get('matches_registered_runtime', True) else ' — 다름'})")
    print(f"  {_mark} {_row['module']:<10} {_row.get('version', '')}{_note}"
          f"  · {_row['purpose']}")
BJ.assert_runtime_ready(MODE)
print("null runner v:", BJ.NULL_RUNNER_VERSION)
print("환경 pin:", BJ.build_env_pin())

# 회귀 테스트를 먼저 돌린다. 여기서 깨지면 아래는 볼 필요가 없다.
proc = subprocess.run([sys.executable,
                       os.path.join(REPO, "mit-bih",
                                    "test_q5d_order_preserving_beat_join.py")],
                      capture_output=True, text=True)
tail = proc.stdout.strip().splitlines()[-1]
print(tail)
assert proc.returncode == 0, proc.stdout[-3000:]
passed = int(tail.split("passed")[1].split("·")[0].strip())
assert passed >= NEED_TESTS, (
    f"회귀 테스트 {passed}개 통과 — 최소 {NEED_TESTS}개를 기대했다. "
    f"낡은 커밋을 보고 있을 수 있다(BRANCH={BRANCH}).")

In [ ]:
# ── 셀 3: Drive mount + 승인 gate ───────────────────────────────────────────
MODE = globals().get("MODE", "DESIGN")
OPEN_REGISTERED_DATA = bool(globals().get("OPEN_REGISTERED_DATA", False))

print(BJ.NO_EXECUTION_BANNER); print()
print(BJ.APPROVAL_NOTE); print()
print("승인 없이 가능 :", BJ.OFFLINE_MODES)
print("opt-in 필요    :", BJ.MODES_NEEDING_EXECUTION_APPROVAL)

APPROVAL = None
if MODE in BJ.MODES_NEEDING_EXECUTION_APPROVAL:
    assert OPEN_REGISTERED_DATA, (
        f"MODE={MODE} 은 등록 자산을 연다. 셀 1 에서 OPEN_REGISTERED_DATA=True 로 "
        f"명시적으로 opt-in 해야 한다. 사용자가 실행을 승인했지만, opt-in 을 "
        f"코드에 남겨 두는 이유는 실수로 데이터를 여는 실행을 막기 위해서다.")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    APPROVAL = BJ.EXECUTION_APPROVAL_TOKEN

print()
print(BJ.design_card(MODE, execution_approved=bool(APPROVAL)))

In [ ]:
# ── 셀 4: 결과 로더 — 모든 결과 셀은 오직 이 함수로만 값을 얻는다 ────────────
import csv, json, os

def bundle_path(name):
    RUN_DIR = globals().get("RUN_DIR", "")
    assert RUN_DIR, ("RUN_DIR 이 비어 있다 — 아직 실행된 run bundle 이 없다.\n"
                     "이 notebook 은 IMPLEMENTED / FULL RESULT NOT RUN 상태다.")
    return os.path.join(RUN_DIR, name)

def load_json(name):
    "결과 숫자는 파일에서 읽는다. 셀 안에서 다시 계산하거나 옮겨 적지 않는다."
    with open(bundle_path(name), encoding="utf-8") as fh:
        return json.load(fh)

def load_csv(name):
    with open(bundle_path(name), encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

def have_results():
    RUN_DIR = globals().get("RUN_DIR", "")
    return bool(RUN_DIR) and os.path.exists(os.path.join(RUN_DIR, "decision.json"))

def need_results(section):
    if not have_results():
        print(f"[{section}] 결과 없음 — FULL RESULT NOT RUN.")
        print("  실행 승인 후 run bundle 이 생기면 이 셀이 그 파일에서 값을 읽는다.")
        return False
    return True

# 이 셀이 도는지 / 왜 안 도는지를 반드시 말한다. 조용히 건너뛴 단계는
# 통과한 단계와 출력이 구분되지 않는다 — 과학 파이프라인에서 가장 위험한
# 실패 방식이라, 건너뛰면 이유와 고치는 법을 찍는다.
def stage_should_run(stage_modes, name):
    mode = globals().get("MODE", "DESIGN")
    approval = globals().get("APPROVAL")
    if mode not in stage_modes:
        print(f"[{name}] SKIP — MODE={mode!r} 이라 이 단계는 돌지 않았다.")
        print(f"  실행하려면 셀 1 에서 MODE = {stage_modes[0]!r} 로 바꾸고"
              f" 셀 1 → 3 → 5 를 다시 실행한 뒤 이 셀을 돌린다.")
        print("  (위에 찍힌 규칙 표는 상수 출력일 뿐 실행 결과가 아니다.)")
        return False
    if not approval:
        print(f"[{name}] SKIP — APPROVAL 이 없다.")
        print("  셀 1 에서 OPEN_REGISTERED_DATA = True 로 두고 셀 3 을 실행한다.")
        return False
    print(f"[{name}] RUN — MODE={mode}")
    return True

print("결과 유무:", have_results())

## 2. 입력 asset/hash 확인 — **preflight STOP/PASS**

`JOIN_INPUT_ABSENT` 는 **자재 계약 실패**일 때만 발화한다 — canonical mamba 자산과
source/meta hash, V9·V10 캐시와 44-record 경계, detection-order 계약,
cache→result NPZ 위치 계약. **join 성능이 나쁘다는 이유로는 이 분기에 오지 않는다.**

셀 5 가 **매칭 이전에** hash 를 돌려 STOP/PASS 를 정한다. STOP 이면 아래 실행 셀은
돌리지 않는다. 특히 `mamba_data.npz` 는 Drive 에 **같은 크기 사본이 3개**라 크기로는
정본을 못 가린다 — `resolve_canonical_mamba()` 가 바이트로 하나만 고르고, 두 개
이상이 맞거나 하나도 안 맞으면 중단한다.

In [ ]:
# ── 셀 5: hash preflight — STOP / PASS (매칭 전에 자재 계약을 닫는다) ───────
MODE = globals().get("MODE", "DESIGN")
print("등록 mamba SHA-256:", BJ.MAMBA_SHA256)
print("등록 mamba Drive id:", BJ.MAMBA_REGISTERED_DRIVE_ID)
print("읽는 result NPZ 키:", BJ.RESULT_NPZ_DS1_AUDIT_KEYS,
      "· 봉인:", BJ.RESULT_NPZ_SEALED)
print("`t` 는 join key 로 쓸 수 없다.")
print("캐시 hash 대상:", len(BJ.cache_expected_files()), "파일 "
      "(meta.json + 44 npz) — 파일명 목록이 아니라 본문 SHA-256 을 집계한다.")
print("MIT-BIH hash 대상:", len(BJ.mitdb_expected_files()),
      "파일 = publisher 트리 48 record x3 + 메타 3 (join 이 읽는 44 가 아니다;"
      " 102/104/107/217 은 paced 라 join 에서 제외되지만 트리에는 존재한다)")
print("  + SHA256SUMS.txt 가 있으면 publisher 체크섬과 직접 대조한다.")
print("result 전수검사  :", len(BJ.result_expected_files(BJ.V10_RESULT_ARMS)),
      "파일 (arm x seed 사전 고정, glob 아님) · pid 만 읽는다")
print("  DS1 은 result 계약을 요구하지 않는다 — cache/ledger 로 검증한다.")
print()

PREFLIGHT = None
CANONICAL_MAMBA = None
if APPROVAL is None:
    print("[2. hash preflight] SKIP — APPROVAL 이 없다.")
    print(f"  MODE={MODE!r} · 이 단계는 셀 1 에서 MODE='HASH_PREFLIGHT' 와"
          f" OPEN_REGISTERED_DATA=True 로 두고 셀 3 을 실행해야 돈다.")
    print("  위 목록은 상수 출력일 뿐 검증 결과가 아니다.")
else:
    PREFLIGHT = BJ.build_preflight(ASSET_MAMBA_CANDIDATES, ASSET_CACHE_V10,
                                   ASSET_MITDB, ASSET_V10_RESULTS, APPROVAL,
                                   BJ.V10_RESULT_ARMS)
    CANONICAL_MAMBA = PREFLIGHT["canonical_mamba"]["path"]

    print("canonical mamba :", CANONICAL_MAMBA)
    print("  등록본 존재   :", PREFLIGHT["canonical_mamba"]["registered_copy_present"])
    print("  byte 동일 사본:", PREFLIGHT["canonical_mamba"]["byte_identical_duplicates"])
    for row in PREFLIGHT["canonical_mamba"]["candidates"]:
        print("   ", row.get("role"), row.get("path"),
              str(row.get("sha256"))[:16])
    for key in ("cache_aggregate", "mitdb_aggregate"):
        agg = PREFLIGHT[key]
        print(f"{key:<18}: {str(agg['aggregate'])[:16]}… "
              f"({agg['n_files']} files · missing {agg['missing']} · "
              f"extra {agg['extra']})")
        pub = agg.get("publisher_checksums")
        if pub:
            print(f"{'':<18}  publisher 체크섬: available={pub['available']} "
                  f"ok={pub['ok']} matched={pub.get('matched')}"
                  f"/{pub.get('checked')} "
                  f"(unlisted {len(pub.get('unlisted') or [])})")
            for miss in pub.get("mismatched") or []:
                print(f"{'':<18}  ! {miss['name']} — join 이 읽는 파일인가: "
                      f"{miss['read_by_the_join']}")
                for k in ("bytes", "non_empty_lines", "has_crlf",
                          "starts_with_bom", "ends_with_newline",
                          "benign_explanation"):
                    if k in miss:
                        print(f"{'':<20}    {k} = {miss[k]}")
                print(f"{'':<20}    published {miss['published_sha256'][:16]}…"
                      f" · observed {miss['observed_sha256'][:16]}…")
                if miss.get("first_lines"):
                    print(f"{'':<20}    first {miss['first_lines']}")
                    print(f"{'':<20}    last  {miss['last_lines']}")
    lc = PREFLIGHT["cache_ledger_contract"]
    print(f"DS1/DS2 cache ledger contract: ok={lc['ok']} {lc['observed']}")
    rc = PREFLIGHT["result_contract"]
    print(f"DS2 result contract ({rc['split']}): ok={rc['ok']} · "
          f"{rc['n_verified']}/{rc['n_expected']} 파일 검증 · "
          f"공통 pid digest {str(rc['pid_digest'])[:16]}")
    for row in rc["files"]:
        if row.get("status") != "VERIFIED":
            print("   !", row.get("name"), row.get("status"))

    print()
    print("PREFLIGHT", "PASS" if PREFLIGHT["ok"] else "STOP")
    for problem in PREFLIGHT["problems"]:
        print("   -", problem)

    # PASS 든 STOP 이든 결과를 Drive 에 보존한다 (STOP 도 기록할 가치가 있다).
    import json, os, time
    ts = time.strftime("%Y%m%dT%H%M%S")
    PREFLIGHT_DIR = f"{DRIVE}/{BJ.DRIVE_RUN_REL}/{ts}_{BJ.RUN_DIR_SUFFIX}_preflight"
    assert not os.path.exists(PREFLIGHT_DIR), "기존 bundle 을 덮어쓰지 않는다"
    os.makedirs(PREFLIGHT_DIR, exist_ok=False)
    with open(f"{PREFLIGHT_DIR}/preflight.json", "w", encoding="utf-8") as fh:
        json.dump(PREFLIGHT, fh, indent=1, ensure_ascii=False)
    with open(f"{PREFLIGHT_DIR}/decision.json", "w", encoding="utf-8") as fh:
        json.dump({"decision": (BJ.DECISION_NOT_RUN if PREFLIGHT["ok"]
                                else BJ.DECISION_INPUT_ABSENT),
                   "stage": "HASH_PREFLIGHT", "ok": PREFLIGHT["ok"],
                   "problems": PREFLIGHT["problems"],
                   "rule_fingerprint": PREFLIGHT["rule_fingerprint"]},
                  fh, indent=1, ensure_ascii=False)
    print("preflight bundle:", PREFLIGHT_DIR)

    # STOP 이면 여기서 멈춘다. 아래 실행 셀은 돌리지 않는다.
    BJ.assert_preflight_passed(PREFLIGHT)

## 3. 44-record ledger 표

record 경계는 **등록된 대장에서 산술로** 잘린다. 라벨이나 join 품질로 추론하지
않는다. 일치 36 record 와 불일치 8 record 는 **사전 등록된 보고 층(strata)** 이고,
둘 다 **같은 matcher** 를 통과한다. 개수가 같다고 위치 동일성으로 간주하지 않는다.

In [ ]:
# ── 셀 6: 44-record ledger (등록 상수에서 생성, 결과와 무관) ─────────────────
report = BJ.verify_ledger()
print("ledger ok        :", report["ok"], report["problems"])
print("records          :", report["records"],
      f"(equal {report['equal_count_records']} · mismatch {report['mismatched_records']})")
print("cache / mamba    :", report["cache_total"], "/", report["mamba_total"],
      "· difference", report["total_difference"])
print()
print(BJ.ledger_table())

## 4. synthetic fixture 결과

합성 fixture 는 **DS1 을 보기 전에** 돈다. false certified pair 가 하나라도 나오면
`JOIN_RULE_FALSIFIED` 로 즉시 종결한다. 이 절은 실행 승인 없이도 돌릴 수 있다 —
등록 자산을 하나도 열지 않기 때문이다.

In [ ]:
# ── 셀 7: fixture 배터리 (합성 데이터만) ────────────────────────────────────
outcomes = BJ.run_synthetic_fixtures()
print(BJ.fixture_card(outcomes))
assert BJ.fixtures_passed(outcomes), "false certified pair 가 있으면 여기서 멈춘다"

## 5. Leg 1 replay audit — `.atr` → mamba

결정론적 **source replay** 다. 통계적 join 이 아니다. 등록된 N/S/V symbol map,
annotation 위치 `pos` 기준 150-sample 경계 규칙, 5개 미만 record 규칙 셋을 원
`.atr` 만으로 재계산하고, record별 개수·순서·RR 을 커밋된 계보 대장과 대조한다.

**첫·끝 beat 는 eligible 하다.** 첫 pre-RR 은 첫 interval 의 복제이고 마지막
post-RR 은 마지막 interval 의 복제다 — 없는 것이 아니다.

불일치는 `JOIN_RULE_FALSIFIED` + `failed_leg = LEG1_SOURCE_REPLAY` 이고 **Leg 2 는
시작하지 않는다.**

In [ ]:
# ── 셀 8: Leg 1 replay audit (실행) ────────────────────────────────────────
print("Leg 1 규칙 (등록 상수):")
print("  symbol map :", {k: v for k, v in sorted(BJ.AAMI_SYMBOL_MAP.items())})
print("  경계 규칙  :", f"{BJ.WIN_BEFORE} <= pos < len(signal) - {BJ.WIN_AFTER}")
print("  record 규칙:", f"유효 beat < {BJ.MIN_VALID_BEATS} 이면 record 통째 제외")
print("  RR         : 필터링 이후, 초 단위, 첫·끝 복제 -> 첫·끝 beat eligible")
print("  기대 총계  : DS1", BJ.REGISTERED_MAMBA_TOTALS["DS1"],
      "· DS2", BJ.REGISTERED_MAMBA_TOTALS["DS2"])
print()
print("V9/V10 캐시의 RR 의미는 다르다 (frontend.py::rr_features):")
print("  endpoint   :", BJ.CACHE_ENDPOINT_SEMANTIC, "(복제가 아니라 0.0)")
print("  계산 시점  :", BJ.CACHE_RR_STAGE)
print("  -> 두 계보의 endpoint 행은 후보 간선을 못 만들고 UNMATCHED 로 남는다."
      " 이건 정직한 결과이고 메우지 않는다.")

if stage_should_run(("LEG1_REPLAY_AUDIT",), "5. Leg 1 replay audit"):
    # Leg 1 도 자재 계약이 닫힌 뒤에만 시작한다. run_join() 은 freeze 를
    # 필수 인자로 받지만 이 감사 셀은 따로 돌 수 있으므로 여기서 막는다.
    assert PREFLIGHT and PREFLIGHT.get("ok"), (
        "셀 5 의 preflight 가 PASS 여야 Leg 1 을 시작한다.")
    assert PREFLIGHT["rule_fingerprint"] == BJ.rule_fingerprint(), (
        "preflight 가 다른 규칙에서 동결됐다 — 셀 5 를 다시 돌린다.")

    leg1 = BJ.replay_leg1_split(ASSET_MITDB, "DS1", APPROVAL)
    mamba = BJ.load_mamba_sequences(CANONICAL_MAMBA, APPROVAL)
    stored = {r: {"pre": list(v.pre_samples), "post": list(v.post_samples)}
              for r, v in mamba["sequences"].items() if v.split == "DS1"}
    report = BJ.audit_leg1_against_ledger(leg1, "DS1", stored, BJ.UNIT_SAMPLES)
    print()
    print("Leg 1 DS1 ok:", report["ok"], "· replayed", report["replayed_total"],
          "/", report["expected_total"])
    for problem in report["problems"][:10]:
        print("   -", problem)
    print("mamba pid 블록이 대장 개수와 맞는가:", mamba["ok"])
    for block in mamba["count_mismatches"][:10]:
        print("   -", block)

    # PASS 든 FALSIFIED 든 보존한다 — 판정은 notebook 출력이 아니라 bundle 이
    # source of truth 다(CLAUDE.md 「실행 로그 루프」).
    import json, os, time
    ts = time.strftime("%Y%m%dT%H%M%S")
    LEG1_DIR = f"{DRIVE}/{BJ.DRIVE_RUN_REL}/{ts}_{BJ.RUN_DIR_SUFFIX}_leg1"
    assert not os.path.exists(LEG1_DIR), "기존 bundle 을 덮어쓰지 않는다"
    os.makedirs(LEG1_DIR, exist_ok=False)
    with open(f"{LEG1_DIR}/leg1_audit.json", "w", encoding="utf-8") as fh:
        json.dump({"report": report, "mamba_blocks": mamba["blocks"],
                   "preflight": {k: PREFLIGHT[k]
                                 for k in BJ.PREFLIGHT_FREEZE_FIELDS}},
                  fh, indent=1, ensure_ascii=False)
    with open(f"{LEG1_DIR}/decision.json", "w", encoding="utf-8") as fh:
        json.dump({"decision": (BJ.DECISION_NOT_RUN if report["ok"]
                                else BJ.DECISION_RULE_FALSIFIED),
                   "stage": "LEG1_REPLAY_AUDIT", "ok": report["ok"],
                   "failed_leg": report["failed_leg"],
                   "problems": report["problems"][:50],
                   "rule_fingerprint": BJ.rule_fingerprint()},
                  fh, indent=1, ensure_ascii=False)
    print("leg1 bundle:", LEG1_DIR)

## 6. Leg 2 record-wise join — mamba → V9/V10 위치 행

검출기 의존이라 `.atr` 로 재계산되지 않는다. 대신 등록 캐시가 행을 물질화해
두었고, record 경계는 대장에서 산술로 나온다.

- V9/V10 행 순서 = `detect_r()` **검출 순서**. `.atr` ordinal 이 아니다.
- result NPZ 는 `prob`·`y`·`pid` 만 저장한다 → identity 는 **위치뿐**이다.
- **전역 정렬 금지.** DS2 의 105·111·222 결손이 이후 모든 record 를 밀어 버린다.
- gap 은 양쪽에서 허용하되 **어떤 행도 impute 하지 않는다.**
- 후보 간선: `|Δpre| <= 1` **그리고** `|Δpost| <= 1` (360 Hz 정수 sample,
  round-half-to-even). 2차 점수·거리 선호·라벨 선호·record별 벌점은 **없다**.
- **CERTIFIED = 모든 maximum-cardinality monotone matching 에 공통으로 든 간선**.
  최적 경로에 따라 달라지는 간선은 `AMBIGUOUS` 이고 unmatched 로 남는다.
  forced edge 는 prefix/suffix DP 로 판정한다 — 최적 매칭을 열거하지 않는다.

In [ ]:
# ── 셀 9: Leg 2 TRUE join — 측정만. 최종 판정이 아니다 ─────────────────────
print("tolerance      :", BJ.RR_TOLERANCE_SAMPLES, "sample @", BJ.FS, "Hz")
print("근거           :", BJ.RR_TOLERANCE_RATIONALE)
print("secondary score: 없음 · distance/label 선호: 없음 · record 벌점: 없음")

if stage_should_run(("LEG2_RECORD_JOIN",), "6. Leg 2 TRUE join"):
    import json, os, time

    assert PREFLIGHT and PREFLIGHT.get("ok"), "preflight PASS 후에만 돈다"

    out = BJ.run_true_join(ASSET_MITDB, CANONICAL_MAMBA, ASSET_CACHE_V10,
                           "DS1", PREFLIGHT, APPROVAL)

    cov = out["coverage"]
    print()
    print("stage :", out["stage_status"], "·", out["gate_status"])
    print(f"overall coverage      : {cov['overall_coverage']:.4f}")
    print(f"class coverage        : "
          + " ".join(f"{c}={cov['class_coverage'][c]:.4f}"
                     for c in BJ.AAMI_CLASSES))
    print(f"class_coverage_balance: {cov['class_coverage_balance']:.4f}")
    print(f"record_coverage_balance:{cov['record_coverage_balance']:.4f}")
    print(f"agreement overall     : {cov['agreement_overall']:.5f}")
    print(f"J_min TRUE            : {out['j_min_true']:.4f}")
    print(f"ambiguous fraction    : {out['ambiguous_fraction']:.4f}")
    print(f"232 S share inflation : "
          f"{out['s_share_inflation'].get('232')}")

    # 진단용 저장만 한다. decision.json 은 만들지 않는다 — null 없이 gate 를
    # 돌리면 12개 중 10개만 보고 JOIN_IDENTIFIABLE 이 나온다. 그건 판정이
    # 아니라 판정처럼 보이는 산출물이고, run bundle 은 영구 보존된다.
    ts = time.strftime("%Y%m%dT%H%M%S")
    TRUE_DIR = f"{DRIVE}/{BJ.DRIVE_RUN_REL}/{ts}_{BJ.RUN_DIR_SUFFIX}_true_join_DS1"
    assert not os.path.exists(TRUE_DIR), "기존 bundle 을 덮어쓰지 않는다"
    os.makedirs(TRUE_DIR, exist_ok=False)
    BJ.write_join_map(out["rows"], f"{TRUE_DIR}/join_map.parquet")
    BJ.write_unmatched_and_ambiguous(out["rows"],
                                     f"{TRUE_DIR}/unmatched_and_ambiguous.csv")
    BJ.write_record_class_coverage(out["coverage"],
                                   f"{TRUE_DIR}/record_class_coverage.csv")
    with open(f"{TRUE_DIR}/true_join.json", "w", encoding="utf-8") as fh:
        json.dump({"stage_status": out["stage_status"],
                   "gate_status": out["gate_status"],
                   "split": out["split"], "coverage": out["coverage"],
                   "s_share_inflation": out["s_share_inflation"],
                   "j_min_true": out["j_min_true"],
                   "ambiguous_fraction": out["ambiguous_fraction"],
                   "per_record_certified": {k: list(v) for k, v in
                                            out["per_record_certified"].items()},
                   "n_processed": out["n_processed"],
                   "leg2_boundaries_ok": out["leg2_boundaries_ok"],
                   "rule_fingerprint": BJ.rule_fingerprint()},
                  fh, indent=1, ensure_ascii=False)
    print()
    print("TRUE join 저장:", TRUE_DIR)
    print("이건 canonical DS1 bundle 이 아니다 — 판정 파일을 만들지 않았다."
          " 최종 판정은 DS1_GATE 가 null 을 결합한 뒤에만 나온다.")

## 6b. 등록 null — shard runner (DS1_GATE)

등록된 null 은 **3 family x 10,000 replicate** 이고 각 replicate 가 **완전한
Leg 2 를 다시** 돈다. 축소·조기중단·근사·family 생략은 허용되지 않는다.
DS1 1회 join 실측 ~1.7 s 기준 30,000 회 = 약 14 시간이라 Colab 세션 하나로는
끝나지 않는다.

그래서 replicate 를 **null shard**(기본 100개 단위)로 나눈다. 과학은 그대로다 —
`apply_control` 이 이미 `(family, replicate)` 로 시드되므로 replicate `b` 의 값은
누가 언제 계산하든 같다. shard 는 **resume artifact** 이고 model checkpoint 가
아니다.

- 각 shard 는 같은 replicate 구간의 **세 family 를 모두** 담는다 → `J_null_max`
  가 shard 경계를 넘지 않는다.
- shard 는 불변이다. 기존 파일을 덮어쓰지 않는다.
- 이미 있는 shard 는 **digest 와 identity(rule fingerprint · code hash ·
  input digest · seed · split)** 를 검증한 뒤에만 재사용한다.
- 중단해도 안전하다. 다시 돌리면 없는 것만 계산한다.
- finalizer 는 0..9999 의 **누락·중복·겹침·digest 불일치·혼합**을 하나라도
  발견하면 STOP 한다. **완결되기 전에는 null_summary·gate 판정·DS2 release 를
  만들지 않는다.**

In [ ]:
# ── 셀 9b: DS1_GATE — TRUE + shard null 을 결합해 canonical bundle 을 만든다 ─
if stage_should_run(("DS1_GATE",), "6b. DS1 gate (sharded null)"):
    import json, os, time

    assert PREFLIGHT and PREFLIGHT.get("ok"), "preflight PASS 후에만 돈다"

    # shard 는 세션을 넘어 재사용돼야 하므로 timestamp 를 쓰지 않는다.
    SHARD_DIR = f"{DRIVE}/{BJ.DRIVE_RUN_REL}/{BJ.RUN_DIR_SUFFIX}_null_shards_DS1"
    # worker 수는 스케줄링이지 과학이 아니다. `worker_count` 는
    # SHARD_DIGEST_FIELDS 에서 의도적으로 빠져 있고, 1 worker 와 2 worker 가
    # 같은 배열·같은 digest 를 낸다는 것을 회귀 테스트가 확인한다
    # (test_worker_count_does_not_change_the_result). 그래서 이 값은 돌리는
    # 머신에 맞춰도 결과가 바뀌지 않는다. 등록 기본값 아래로는 내려가지 않는다.
    DETECTED_CPUS = os.cpu_count() or 1
    MAX_WORKERS = max(BJ.DEFAULT_MAX_WORKERS, DETECTED_CPUS)
    SHARD_SIZE = BJ.DEFAULT_SHARD_SIZE

    est = BJ.estimate_null_runtime(1.7)
    print(f"등록 null: {est['joins']:,} 회 완전 재매칭 ≈ {est['total_hours']:.0f} 시간")
    print("  " + est["note"])
    print(f"  worker {MAX_WORKERS} 기준 예상 wall-clock ≈ "
          f"{est['total_hours'] / MAX_WORKERS:.1f} 시간")
    print(f"shard {len(BJ.shard_plan(BJ.N_NULL_REPLICATES, SHARD_SIZE))} 개"
          f" x {SHARD_SIZE} replicate · worker {MAX_WORKERS}"
          f" (vCPU {DETECTED_CPUS}, worker 당 RSS ≈ 0.5 GB)")
    print(f"resume: {SHARD_DIR}")
    print("세션이 끊기면 이 셀을 다시 실행한다 — 끝난 shard 는 검증 후 재사용된다.")

    result = BJ.run_ds1_gate_sharded(
        ASSET_MITDB, CANONICAL_MAMBA, ASSET_CACHE_V10, PREFLIGHT, SHARD_DIR,
        APPROVAL, shard_size=SHARD_SIZE, max_workers=MAX_WORKERS,
        git_commit=COMMIT, progress=print)

    # 여기까지 왔다는 것은 null 이 3 x 10,000 으로 완결됐다는 뜻이다.
    # 이제서야 canonical bundle 을 만든다.
    ts = time.strftime("%Y%m%dT%H%M%S")
    RUN_DIR = f"{DRIVE}/{BJ.DRIVE_RUN_REL}/{ts}_{BJ.RUN_DIR_SUFFIX}_DS1_GATE"
    assert not os.path.exists(RUN_DIR), "기존 bundle 을 덮어쓰지 않는다"
    os.makedirs(RUN_DIR, exist_ok=False)

    true_out = result["true"]
    BJ.write_join_map(true_out["rows"], f"{RUN_DIR}/join_map.parquet")
    BJ.write_unmatched_and_ambiguous(true_out["rows"],
                                     f"{RUN_DIR}/unmatched_and_ambiguous.csv")
    BJ.write_record_class_coverage(true_out["coverage"],
                                   f"{RUN_DIR}/record_class_coverage.csv")
    BJ.write_synthetic_fixture_results(BJ.run_synthetic_fixtures(),
                                       f"{RUN_DIR}/synthetic_fixture_results.csv")
    for name, payload in (
            ("config.json", BJ.build_config("DS1_GATE", ts, True)),
            ("manifest.json", BJ.build_manifest({"mamba": CANONICAL_MAMBA}, ts,
                                                PREFLIGHT)),
            ("decision.json", result["decision"]),
            ("null_summary.json", result["null"]),
            ("bootstrap.json", result["bootstrap"])):
        with open(f"{RUN_DIR}/{name}", "w", encoding="utf-8") as fh:
            json.dump(payload, fh, indent=1, ensure_ascii=False)
    with open(f"{RUN_DIR}/log.txt", "w", encoding="utf-8") as fh:
        fh.write(f"{BJ.NO_EXECUTION_BANNER}\n{BJ.APPROVAL_NOTE}\n")
        fh.write(f"shards={result['shards']} "
                 f"null={len(result['null']['j_null_max'])} "
                 f"bootstrap={result['bootstrap']['replicates']}\n")
    with open(f"{RUN_DIR}/summary.md", "w", encoding="utf-8") as fh:
        fh.write(f"# Q5-D DS1 gate\n\n"
                 f"- decision: `{result['decision']['decision']}`\n"
                 f"- first stopping reason: "
                 f"`{result['decision']['first_stopping_reason']}`\n"
                 f"- gates: {result['decision']['gates_passed']}"
                 f"/{result['decision']['gates_total']}\n"
                 f"- J_min TRUE: {result['decision']['j_min_true']}\n"
                 f"- null: {len(result['null']['j_null_max'])} x "
                 f"{len(BJ.CONTROL_FAMILIES)} families\n"
                 f"- bootstrap: {result['bootstrap']['replicates']}\n\n"
                 f"V10 probability 를 열지 않았고 association 을 하지 않았다.\n")

    print()
    print("판정:", result["decision"]["decision"])
    print("canonical DS1 bundle:", RUN_DIR)
    print("DS2 release 는 이 bundle 만 입력으로 인정한다.")

## 7. DS1 gate report

12개 gate 전부 통과해야 규칙이 자격을 얻는다. 90% pooled coverage 로는 부족하다는
Q5-B-0 의 교훈이 class·record 하위 꼬리 gate 로 남아 있다.

In [ ]:
# ── 셀 10: DS1 gate 표 — decision.json 에서 읽는다 ──────────────────────────
print("등록 임계값:")
for name, value in (("3 overall coverage", BJ.GATE_COVERAGE_MIN),
                    ("4 S coverage", BJ.GATE_S_COVERAGE_MIN),
                    ("5 per-class coverage", BJ.GATE_PER_CLASS_COVERAGE_MIN),
                    ("6 class balance", BJ.GATE_CLASS_BALANCE_MIN),
                    ("7 record coverage", BJ.GATE_RECORD_COVERAGE_MIN),
                    ("7 record balance", BJ.GATE_RECORD_BALANCE_MIN),
                    ("8 agreement overall", BJ.GATE_AGREEMENT_OVERALL_MIN),
                    ("8 agreement per class", BJ.GATE_AGREEMENT_PER_CLASS_MIN),
                    ("10 signal/null", BJ.GATE_SIGNAL_TO_NULL_MIN),
                    ("12 S share inflation", BJ.GATE_S_SHARE_INFLATION_MAX)):
    print(f"  {name:<24} {value}")

if need_results("7. DS1 gate report"):
    decision = load_json("decision.json")
    print(f"\n{'gate':<28} {'pass':<6} value / threshold")
    for gate in decision["gates"]:
        print(f"  {gate['gate']:<26} {str(gate['passed']):<6} "
              f"{gate['value']} / {gate['threshold']}")
    print("\n통과:", decision["gates_passed"], "/", decision["gates_total"])

## 8. DS2 frozen gate report

DS1 규칙·source hash·테스트·환경·임계값·DS1 보고서가 **동결된 뒤에만** 같은 지지
gate(2-8, 12)를 DS2 에 **한 번** 적용한다. DS2 는 null 을 다시 돌리지 않고 어떤
상수도 바꾸지 않는다. **DS2 gate 가 실패하면 `JOIN_SELECTION_BIASED` 로 멈추고
V10 probability 를 열지 않는다.**

DS2 per-beat class label 은 이 단계 전까지 봉인이며, **join 규칙 선택에는 절대
쓰이지 않는다.**

In [ ]:
# ── 셀 11: DS2 지지 gate (동결 후 1회) ──────────────────────────────────────
print("DS2 에 적용되는 gate: 2-8 과 12 (null 재실행 없음, 상수 변경 없음)")
print("DS2 label 봉인 해제는 별도 토큰이 필요하다:", BJ.DS2_LABEL_RELEASE_FLAG)

if need_results("8. DS2 frozen gate report"):
    decision = load_json("decision.json")
    ds2 = [g for g in decision["gates"] if g.get("detail", "").startswith("DS2")]
    for gate in ds2 or decision["gates"]:
        print(f"  {gate['gate']:<26} passed={gate['passed']} value={gate['value']}")

## 9. negative-control / null plots

세 음성대조군은 각각 **완전한 Leg 2 를 다시 돌린다** — 후보 간선 구성, record별
최대 매칭, certification, 감사 통계 전부. `SEQUENCE_RELATIONSHIP` 외에는 아무것도
바뀌지 않는다.

`J_null_max[b] = max(J_wrong[b], J_shuffle[b], J_shift[b])` · master seed `2026017` ·
10,000 replicate. bootstrap 은 record-cluster 2,000 replicate, seed `2026018`.

**규칙이 바뀌면 저장된 null 을 재사용할 수 없다** — `rule_fingerprint` 가 함께
움직이고 `assert_null_matches_rule()` 이 거부한다.

In [ ]:
# ── 셀 12: TRUE J_min 대 max-null 분포 ──────────────────────────────────────
print("families:", BJ.CONTROL_FAMILIES)
print("seeds   :", BJ.MASTER_SEED, "/", BJ.BOOTSTRAP_SEED)
print("replicates:", BJ.N_NULL_REPLICATES, "/", BJ.N_BOOTSTRAP_REPLICATES)

if need_results("9. negative-control / null"):
    import matplotlib.pyplot as plt
    null = load_json("null_summary.json")
    BJ.assert_null_matches_rule(null)          # 완화된 규칙이 물려받지 못하게
    boot = load_json("bootstrap.json")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(null["j_null_max"], bins=60, color="#9aa5b1",
            label="max-null $J_{null,max}$")
    ax.axvline(null["q95"], color="#e08a00", ls="--", label=f"q95 {null['q95']:.4f}")
    ax.axvline(null["q99"], color="#c23b22", ls="--", label=f"q99 {null['q99']:.4f}")
    ax.axvline(null["j_true"], color="#1f6feb", lw=2,
               label=f"TRUE $J_{{min}}$ {null['j_true']:.4f}")
    ax.set_xlabel("$J_{min}$"); ax.set_ylabel("replicates")
    ax.set_title("TRUE $J_{min}$ vs family-wise max-null")
    ax.legend(); fig.tight_layout(); plt.show()

    print("signal_to_null:", null["signal_to_null"],
          ">= ", BJ.GATE_SIGNAL_TO_NULL_MIN)
    print("bootstrap 95% CI of (J_min_TRUE - q95):",
          boot["ci_low"], boot["ci_high"])

## 10. class·record coverage plots

`processed` 는 언제나 **V9/V10 위치 행**이지 mamba 행이 아니다. coverage 의 분모는
그 행이다.

In [ ]:
# ── 셀 13: record별 certified coverage · N/S/V class coverage · balance ─────
if need_results("10. coverage plots"):
    import matplotlib.pyplot as plt
    rows = load_csv("record_class_coverage.csv")
    per_record = [(r["record"], float(r["certified_coverage"]))
                  for r in rows if not r["record"].startswith("__class_")]
    per_class = [(r["record"].replace("__class_", ""),
                  float(r["certified_coverage"]))
                 for r in rows if r["record"].startswith("__class_")]
    decision = load_json("decision.json")
    gates = {g["gate"]: g for g in decision["gates"]}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2),
                             gridspec_kw={"width_ratios": [3, 1, 1]})
    axes[0].bar([r for r, _v in per_record], [v for _r, v in per_record],
                color="#1f6feb")
    axes[0].axhline(BJ.GATE_RECORD_COVERAGE_MIN, color="#c23b22", ls="--",
                    label=f"record floor {BJ.GATE_RECORD_COVERAGE_MIN}")
    axes[0].set_title("record별 certified coverage")
    axes[0].tick_params(axis="x", rotation=90); axes[0].legend()

    axes[1].bar([c for c, _v in per_class], [v for _c, v in per_class],
                color=["#5a7d9a", "#e08a00", "#7a5195"])
    axes[1].axhline(BJ.GATE_PER_CLASS_COVERAGE_MIN, color="#c23b22", ls="--")
    axes[1].set_title("N/S/V class coverage")

    balance = [("class_coverage_balance", gates["6_class_coverage_balance"]["value"]),
               ("record_coverage_balance",
                gates["7_record_coverage"]["value"]["balance"])]
    axes[2].bar([b for b, _v in balance], [v for _b, v in balance],
                color="#2f7d32")
    axes[2].axhline(BJ.GATE_CLASS_BALANCE_MIN, color="#c23b22", ls="--")
    axes[2].set_title("balance gates"); axes[2].tick_params(axis="x", rotation=20)
    fig.tight_layout(); plt.show()

## 11. ambiguous / unmatched 원인표

`AMBIGUOUS` 는 **실패가 아니라 정직한 보고**다. 여러 최적 경로에 걸쳐 달라지는
간선은 certify 하지 않고 unmatched 로 남긴다. 반복 RR 때문에 너무 많이 남으면
그것이 `JOIN_UNRESOLVED` — "식별 가능한 것이 없다" 는 유효한 결과다.

Leg 1 실패와 Leg 2 실패는 **분리해서** 센다.

In [ ]:
# ── 셀 14: 원인별 개수 + Leg 1 / Leg 2 실패 분해 ───────────────────────────
if need_results("11. ambiguous/unmatched 원인표"):
    import collections
    import matplotlib.pyplot as plt
    rows = load_csv("unmatched_and_ambiguous.csv")
    reasons = collections.Counter(r["drop_or_unmatched_reason"] for r in rows)
    legs = collections.Counter(r["failed_leg"] or "none" for r in rows)

    print(f"{'reason':<40} count")
    for reason, count in reasons.most_common():
        print(f"  {reason:<38} {count}")
    print()
    print("leg별 실패 수:", dict(legs))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].bar(list(reasons), list(reasons.values()), color="#8a6d3b")
    axes[0].set_title("ambiguity / unmatched reason counts")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(list(legs), list(legs.values()), color="#c23b22")
    axes[1].set_title("Leg 1 · Leg 2 별 실패 수")
    fig.tight_layout(); plt.show()

## 12. equal-count 36 vs mismatch 8 진단 비교

**진단용 층일 뿐이다.** 어느 층도 제외하거나 다른 matcher 를 주거나 실패한 primary
gate 를 구제하는 데 쓸 수 없다. 개수가 같다는 것은 위치 동일성이 아니다 — mamba 는
주석 위치 `pos`, V9/V10 은 검출 위치 `p` 로 경계를 자르므로 drop-one/add-one 상쇄가
가능하다.

In [ ]:
# ── 셀 15: 두 층 비교 ───────────────────────────────────────────────────────
ledger = BJ.build_ledger()
mismatched = [(s, r.record, r.delta) for s in BJ.SPLITS for r in ledger[s]
              if r.stratum == BJ.STRATUM_MISMATCH]
print("사전 등록된 불일치 record:", mismatched)
print("두 층 모두 동일한 matcher 를 통과한다.")

if need_results("12. equal-count 36 vs mismatch 8"):
    import matplotlib.pyplot as plt
    strata = load_json("decision.json").get("strata", {})
    if strata:
        names = list(strata)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(names, [strata[n]["coverage"] for n in names],
               color=["#1f6feb", "#e08a00"])
        ax.axhline(BJ.GATE_COVERAGE_MIN, color="#c23b22", ls="--")
        ax.set_title("equal-count 36 vs mismatch 8 — certified coverage")
        fig.tight_layout(); plt.show()
        for name in names:
            print(f"  {name:<18} {strata[name]}")

## 13. record 105 · 111 · 116 · 208 · 222 상세표

개수가 어긋나는 record 들이다. **전역 정렬이 왜 금지인지**를 보여 주는 자리이기도
하다 — DS2 의 105·111·222 결손은 전역 정렬에서 이후 모든 record 를 밀어 버린다.

`222` 는 자격검증에서 PPV 0.4873 으로 per-record floor 를 못 넘겼지만(주석 1,257 대
검출 2,477 → PPV 상한 0.5075) **primary 에서 제외·가중·보정하지 않는다.** 낮은
전문가-참조 PPV 가 올바른 beat join 을 실패시키게 두지 않고, 좋은 beat join 으로
미주석 P 검출의 생물학적 의미를 결론짓지도 않는다.

In [ ]:
# ── 셀 16: 불일치 record 상세 ───────────────────────────────────────────────
WATCH = ("105", "111", "116", "208", "222")
print(f"{'split':<5} {'rec':>4} {'cache_n':>8} {'mamba_n':>8} {'diff':>5} "
      f"{'cache@':>8} {'mamba@':>8}")
for split in BJ.SPLITS:
    for row in BJ.build_ledger()[split]:
        if row.record in WATCH:
            print(f"{split:<5} {row.record:>4} {row.cache_n:>8} {row.mamba_n:>8} "
                  f"{row.delta:>5} {row.cache_start:>8} {row.mamba_start:>8}")

if need_results("13. record 상세표"):
    rows = load_csv("record_class_coverage.csv")
    for row in rows:
        if row["record"] in WATCH:
            print(f"  {row['record']}  coverage={row['certified_coverage']}")

## 14. record 232 — S-share 원분포 대 certification 후 inflation

**join 이전에 이미** DS2 S beat 1,837 중 record `232` 가 1,382 개, **75.2%** 다.
이것은 **source concentration** 이고, 유리한 join 부분집합을 골라도 고쳐지지 않는다.

join gate 12 의 `S_share_inflation` 은 **certification 이 그 편중을 더 키웠는지**만
본다(≤ 1.25). parent spec 의 **절대 50% ceiling 을 완화하거나 대체하지 않는다.**

→ **join 이 성공해도 parent association 은 자기 gate 때문에 막힐 수 있다.** 그
해소는 별도 parent-spec 개정이 필요하고, 이 join 설계는 그런 개정을 하지 않는다.

In [ ]:
# ── 셀 17: 232 의 원 share 대 certified share ──────────────────────────────
print(f"source: record 232 = {BJ.RECORD_232_S_BEATS}/{BJ.DS2_S_BEATS_TOTAL} "
      f"DS2 S beats = {BJ.RECORD_232_S_SHARE:.4f}")
print(f"parent 절대 ceiling : {BJ.PARENT_ABSOLUTE_RECORD_S_SHARE_CEILING} "
      f"→ 이미 초과 (join 이전부터)")
print(f"join gate 12 ceiling: inflation <= {BJ.GATE_S_SHARE_INFLATION_MAX} "
      f"(source 대비 비율, 절대 share 가 아니다)")

if need_results("14. record 232"):
    import matplotlib.pyplot as plt
    shares = load_json("decision.json").get("s_share", {})
    if shares:
        records = sorted(shares)
        fig, ax = plt.subplots(figsize=(9, 4))
        width = 0.4
        idx = range(len(records))
        ax.bar([i - width / 2 for i in idx],
               [shares[r]["source_share"] for r in records], width,
               label="source share", color="#9aa5b1")
        ax.bar([i + width / 2 for i in idx],
               [shares[r]["certified_share"] for r in records], width,
               label="certified share", color="#1f6feb")
        ax.set_xticks(list(idx)); ax.set_xticklabels(records, rotation=90)
        ax.set_title("record 232 포함 — S source share vs certified share")
        ax.legend(); fig.tight_layout(); plt.show()
        print("232 inflation:", shares.get("232", {}).get("inflation"))

## 15. decision tree 최종 판정

first-failure-wins 로 **하나의 primary decision** 만 기록한다. 그러나 모든 audit
gate 의 수치와 pass/fail 은 따로 전부 저장한다 — 첫 실패 뒤에 통과한 gate 도
남는다.

In [ ]:
# ── 셀 18: 최종 판정 ────────────────────────────────────────────────────────
print("판정 후보:", BJ.DECISIONS)
print()
if not need_results("15. decision tree"):
    print(json.dumps(BJ.not_run_decision("implementation only; execution "
                                         "awaiting the separate approval"),
                     indent=2, ensure_ascii=False))
else:
    decision = load_json("decision.json")
    print("decision              :", decision["decision"])
    print("first_stopping_reason :", decision["first_stopping_reason"])
    print("failed_leg            :", decision["failed_leg"])
    print("gates                 :", decision["gates_passed"], "/",
          decision["gates_total"])
    for flag in ("training_performed", "model_scored",
                 "v10_probability_opened", "association_performed"):
        print(f"  {flag:<24}", decision.get(flag))

## 16. 사람이 읽을 수 있는 해석 요약

이 절은 `summary.md` 를 그대로 보여 준다. **숫자를 여기에 옮겨 적지 않는다.**

읽을 때 함께 기억할 것:

- `JOIN_IDENTIFIABLE` 은 **beat identity map 을 만들어도 된다**는 뜻일 뿐이다.
  P-timing association 이 아니고, V10 probability 를 여는 승인도 아니다.
- `JOIN_UNRESOLVED` 는 실패가 아니라 **"현재 산출물로는 식별 불가"** 라는 유효한
  결과다. Q5-B-0 이 남긴 교훈대로, 많이 붙었다와 쓸 수 있다는 다르다.
- 자격검증은 per-record floor **정확히 5/6** 으로 통과했다 — 여유가 0이었다.
  join 이 이것을 "측정 품질이 균일하다" 는 증거로 승격시킬 수 없다.

In [ ]:
# ── 셀 19: summary.md 그대로 출력 ──────────────────────────────────────────
if need_results("16. 해석 요약"):
    with open(bundle_path("summary.md"), encoding="utf-8") as fh:
        print(fh.read())

## 17. Drive bundle 저장 및 ingest 단계

**이 notebook 은 run bundle 을 만들지 않는다.** 실제 실행 승인 이후에만 아래 12개
파일이 생기고, 기존 자산을 덮어쓰지 않는다. `join_map` 에는 V10 probability 값을
저장하지 않는다.

실행 후 순서: ① Drive 에 bundle 저장 → ② 실행된 notebook 커밋 →
③ `python pipelines/ingest_run.py --results result.json --notebook notebooks/…ipynb`
로 실행 로그 카드 생성(수치는 실측, LLM 이 지어내지 않는다).

In [ ]:
# ── 셀 20: bundle 계약 확인 (생성하지 않는다) ──────────────────────────────
print("run 디렉터리:", f"MyDrive/{BJ.DRIVE_RUN_REL}/<timestamp>_{BJ.RUN_DIR_SUFFIX}/")
print("필수 파일:")
for name in BJ.BUNDLE_FILES:
    print("  -", name)
print()
print("join_map 열:", BJ.JOIN_MAP_FIELDS)
print("join_map 금지 열:", BJ.JOIN_MAP_BANNED_FIELDS)
print()
if have_results():
    complete, missing = BJ.bundle_is_complete(globals().get("RUN_DIR", ""))
    print("bundle 완전:", complete, "· 누락:", missing)
else:
    print("지금은 bundle 이 없다 — IMPLEMENTED / FULL RESULT NOT RUN.")
    print("Drive 변경 0건 · 확률 NPZ 열람 0건 · DS2 label 열람 0건.")